# Cuaderno 02c: Vectorización de Texto con TF-IDF y N-gramas (Fase 5 - NLP)

- **Estudiante:** Ramiro
- **Rol:** Responsable de la Fase 5 de NLP (Vectorización) y la Capa Backend
- **Materia:** Tópicos y Programación Avanzada
- **Proyecto:** Asistente Virtual Inteligente para Tiendas de Ropa

---

## 1. Contexto y Objetivos del Cuaderno

El propósito de este cuaderno es analizar, justificar y evaluar la **Fase 5 del Pipeline de Procesamiento de Lenguaje Natural**: la conversión del texto en representaciones numéricas mediante **TF-IDF (Term Frequency - Inverse Document Frequency)** y **N-gramas**.

### Objetivos Específicos:
1. **Fundamentación matemática:** Justificar cómo opera el cálculo de $TF$, $IDF$, la atenuación sublineal y la normalización $L_2$.
2. **Comparativa empírica de N-gramas:** Demostrar por qué el rango `(1, 2)` (unigramas + bigramas) supera a `(1, 1)` (solo unigramas) en consultas del dominio de tiendas de ropa.
3. **Justificación de parámetros:** Documentar el propósito de `min_df`, `max_df`, `max_features` y `sublinear_tf`.
4. **Validación en clasificación:** Evaluar el impacto de la vectorización en la clasificación de intenciones mediante validación cruzada estratificada ($F_1$-Macro y Accuracy).

## 2. Marco Teórico: Representación Vectorial con TF-IDF

A diferencia de un conteo simple de palabras (*CountVectorizer*), TF-IDF equilibra la frecuencia local de un término con su rareza global a través de todo el corpus:

### 1. Frecuencia de Término Sublineal ($TF$):
$$\text{TF}(t, d) = 1 + \ln(\text{count}(t, d)) \quad \text{si } \text{count}(t, d) > 0$$
*Justificación:* Si la palabra 'jean' aparece 3 veces en una misma consulta, no aporta el triple de información que si aparece 1 vez. La escala logarítmica atenúa repeticiones artificiales.

### 2. Frecuencia Inversa de Documento ($IDF$ Suavizado):
$$\text{IDF}(t, D) = \ln\left(\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|}\right) + 1$$
*Justificación:* Penaliza términos ubicuos y premia términos específicos y discriminantes de intención (ej. 'reembolso', 'talla', 'envío').

### 3. Normalización Euclidiana ($L_2$):
$$\vec{v}_d = \frac{\vec{v}_d}{\|\vec{v}_d\|_2}$$
*Justificación:* Asegura que frases largas y cortas tengan la misma escala vectorial para que la longitud de la consulta no distorsione la clasificación.

In [ ]:
# Inicialización de dependencias y configuración de entorno
import sys
from pathlib import Path

# Asegurar que la raíz del proyecto esté en el PYTHONPATH
raiz_proyecto = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(raiz_proyecto) not in sys.path:
    sys.path.insert(0, str(raiz_proyecto))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

from src.config import config
from src.nlp.vectorizacion import crear_vectorizador, comparar_vocabulario_ngramas

# Configuración visual de gráficos para Jupyter
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print('Entorno inicializado correctamente.')
print('Proyecto:', config['proyecto']['nombre'])
print('Configuración del vectorizador en config.yaml:', config['vectorizador'])

## 3. Dataset del Dominio: Frases Típicas para Tienda de Ropa

Cargamos un corpus estructurado con las **13 intenciones del proyecto** (9 de clientes y 4 de administración) para evaluar el comportamiento léxico del vectorizador.

In [ ]:
# Dataset de consultas representativas del dominio de tiendas de ropa
datos_corpus = [
    # 1. consultar_stock
    ("tienen jeans negros en talla 32", "consultar_stock"),
    ("hay stock de polera blanca en talla m", "consultar_stock"),
    ("les queda chaqueta de cuero en talla l", "consultar_stock"),
    ("tienen pantalones de vestir disponibles", "consultar_stock"),
    ("busco falda azul en talla s", "consultar_stock"),
    ("tienen vestido rojo talla 38", "consultar_stock"),
    ("queda stock de chamarra negra", "consultar_stock"),
    ("tienen abrigo largo en talla xl", "consultar_stock"),
    # 2. consultar_precio
    ("cuanto cuesta la chaqueta de cuero", "consultar_precio"),
    ("que precio tiene el pantalon de mezclilla", "consultar_precio"),
    ("cual es el valor de la polera con estampado", "consultar_precio"),
    ("cuanto vale este vestido floreado", "consultar_precio"),
    ("precio de los botines de dama", "consultar_precio"),
    ("a cuanto estan las camisas de algodon", "consultar_precio"),
    # 3. consultar_tallas
    ("que tallas tienen disponibles de este pantalon", "consultar_tallas"),
    ("viene en talla xl o solo s y m", "consultar_tallas"),
    ("que medidas tiene la talla mediana", "consultar_tallas"),
    ("tienen tabla de tallas para camisas", "consultar_tallas"),
    ("que tallas manejan en zapatillas", "consultar_tallas"),
    # 4. envios
    ("hacen envios a provincia o todo el pais", "envios"),
    ("cuanto cuesta el delivery a mi domicilio", "envios"),
    ("cuanto tarda en llegar el pedido a la paz", "envios"),
    ("tienen envio gratis por compras mayores", "envios"),
    ("que empresa de encomiendas utilizan para envios", "envios"),
    # 5. devoluciones
    ("quiero cambiar una prenda que me quedo chica", "devoluciones"),
    ("cual es la politica de devolucion si no me gusta", "devoluciones"),
    ("puedo pedir reembolso de mi compra", "devoluciones"),
    ("cuantos dias tengo para hacer un cambio", "devoluciones"),
    ("se puede devolver una prenda comprada en oferta", "devoluciones"),
    # 6. horarios_ubicacion
    ("a que hora abren la tienda en la manana", "horarios_ubicacion"),
    ("donde queda ubicada la sucursal central", "horarios_ubicacion"),
    ("atienden los domingos y feriados", "horarios_ubicacion"),
    ("cual es la direccion exacta de su tienda fisica", "horarios_ubicacion"),
    # 7. metodos_pago
    ("aceptan transferencias bancarias o qr simple", "metodos_pago"),
    ("se puede pagar con tarjeta de credito visa", "metodos_pago"),
    ("aceptan pago contra entrega en efectivo", "metodos_pago"),
    ("tienen descuento por pago al contado", "metodos_pago"),
    # 8. saludo
    ("hola buenas tardes como estan", "saludo"),
    ("buenos dias quisiera hacer una consulta", "saludo"),
    ("hola que tal amigos", "saludo"),
    # 9. despedida
    ("muchas gracias eso era todo hasta luego", "despedida"),
    ("gracias por la atencion adios", "despedida"),
    ("hasta pronto muy amables", "despedida"),
    # 10. reporte_ventas (admin)
    ("dame el reporte de ventas de esta semana", "reporte_ventas"),
    ("cuanto se vendio en total el mes pasado", "reporte_ventas"),
    ("generar balance de ventas de hoy", "reporte_ventas"),
    ("resumen de ingresos de la tienda", "reporte_ventas"),
    # 11. productos_mas_vendidos (admin)
    ("cuales fueron los articulos mas vendidos de octubre", "productos_mas_vendidos"),
    ("top 5 prendas con mayor demanda este mes", "productos_mas_vendidos"),
    ("que ropa es la que mas se compra", "productos_mas_vendidos"),
    # 12. stock_bajo (admin)
    ("que prendas tienen stock critico o bajo", "stock_bajo"),
    ("alerta de productos que estan por agotarse", "stock_bajo"),
    ("inventario que necesita reposicion urgente", "stock_bajo"),
    # 13. ventas_por_categoria (admin)
    ("ventas agrupadas por categoria de prenda", "ventas_por_categoria"),
    ("rendimiento de ventas en jeans vs vestidos", "ventas_por_categoria"),
    ("desglose de ingresos por seccion de ropa", "ventas_por_categoria")
]

df = pd.DataFrame(datos_corpus, columns=["texto", "intencion"])
print(f"Total de consultas en el corpus: {len(df)}")
print(f"Cantidad de intenciones evaluadas: {df['intencion'].nunique()}")
df.head(6)

## 4. Experimento 1: Unigramas (1, 1) vs. Unigramas + Bigramas (1, 2)

Comparamos cuantitativamente la matriz dispersa generada considerando únicamente palabras individuales frente a la inclusión de pares contiguos.

In [ ]:
# Comparación usando la función modular de src/nlp/vectorizacion.py
res_comp = comparar_vocabulario_ngramas(df['texto'].tolist())

print("=== COMPARACIÓN DE VOCABULARIO (Fase 5) ===")
print(f"Columnas con solo Unigramas (1, 1)         : {res_comp['num_unigramas']}")
print(f"Columnas con Unigramas + Bigramas (1, 2)   : {res_comp['num_total_con_bigramas']}")
print(f"Cantidad de bigramas agregados             : {res_comp['num_bigramas_agregados']}")
print(f"Incremento del espacio vectorial           : {res_comp['num_bigramas_agregados'] / res_comp['num_unigramas'] * 100:.1f}%")

print("\nMuestra de bigramas capturados con alto valor semántico:")
for bg in res_comp['ejemplos_bigramas'][:12]:
    print(f"  - '{bg}'")

In [ ]:
# Gráfico comparativo de tamaño de vocabulario
fig, ax = plt.subplots(figsize=(7, 4))
categorias = ['Unigramas (1, 1)', 'Unigramas + Bigramas (1, 2)']
valores = [res_comp['num_unigramas'], res_comp['num_total_con_bigramas']]
colores = ['#2980b9', '#27ae60']

barras = ax.bar(categorias, valores, color=colores, width=0.45, edgecolor='black', linewidth=1)
ax.set_ylabel('Cantidad de Columnas (Términos en Vocabulario)')
ax.set_title('Expansión del Vocabulario por Inclusión de Bigramas')

for bar in barras:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, h + 3, f"{int(h)} términos", ha='center', va='bottom', fontweight='bold')

# Guardar figura para el informe técnico
ruta_img1 = raiz_proyecto / "artifacts" / "vocabulario_ngramas.png"
ruta_img1.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(ruta_img1, dpi=120, bbox_inches="tight")
plt.tight_layout()
print(f"Gráfico guardado en: {ruta_img1}")

### Análisis del Experimento 1
- La inclusión de bigramas captura términos compuestos indispensables en ropa:
  - **`'talla 32'`**, **`'talla m'`**, **`'talla xl'`** $\rightarrow$ Vinculan la dimensión exacta a la prenda.
  - **`'chaqueta de'`**, **`'vestido floreado'`** $\rightarrow$ Especificidad de producto.
  - **`'envio gratis'`**, **`'cuanto cuesta'`**, **`'stock de'`** $\rightarrow$ Señales determinantes para clasificar la intención.

## 5. Experimento 2: Justificación de Hiperparámetros (`min_df`, `max_df`, `sublinear_tf`)

Analizamos cómo afectan los parámetros documentados en `src/nlp/vectorizacion.py`:

In [ ]:
# Comparación de filtrado de ruido con min_df
v_min1 = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
v_min2 = TfidfVectorizer(ngram_range=(1, 2), min_df=2)

X1 = v_min1.fit_transform(df['texto'])
X2 = v_min2.fit_transform(df['texto'])

print("=== IMPACTO DE MIN_DF ===")
print(f"Vocabulario con min_df=1: {X1.shape[1]} características (conserva términos únicos)")
print(f"Vocabulario con min_df=2: {X2.shape[1]} características (descarta términos que aparecen 1 sola vez)")
print(f"Reducción de dimensionalidad: {(1 - X2.shape[1]/X1.shape[1])*100:.1f}%")

# Demostración matemática del escalado sublinear_tf (1 + ln(TF))
texto_repetitivo = ["jean jean jean negro talla 32"]
v_lineal = TfidfVectorizer(sublinear_tf=False, norm=None)
v_sub = TfidfVectorizer(sublinear_tf=True, norm=None)

peso_lin = v_lineal.fit_transform(texto_repetitivo).toarray()[0][v_lineal.vocabulary_['jean']]
peso_sub = v_sub.fit_transform(texto_repetitivo).toarray()[0][v_sub.vocabulary_['jean']]

print("\n=== IMPACTO DE SUBLINEAR_TF ===")
print(f"Palabra 'jean' repetida 3 veces con TF Lineal    : {peso_lin:.2f}")
print(f"Palabra 'jean' repetida 3 veces con TF Sublineal : {peso_sub:.2f} (1 + ln(3))")

## 6. Experimento 3: Validación Empírica en Clasificación de Intenciones

Demostramos el impacto en las métricas finales ($F_1$-Macro y Accuracy) evaluando con **Regresión Logística** y **ComplementNB** bajo validación cruzada estratificada de 5 particiones (`StratifiedKFold(n_splits=5)`).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=config['proyecto']['semilla'])

# Pipeline 1: Unigramas + Regresión Logística
p_uni_lr = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 1), sublinear_tf=True)),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

# Pipeline 2: Unigramas + Bigramas + Regresión Logística (Configuración de Producción)
p_bi_lr = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

# Pipeline 3: Unigramas + ComplementNB
p_uni_nb = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 1), sublinear_tf=True)),
    ('clf', ComplementNB(alpha=0.5))
])

# Pipeline 4: Unigramas + Bigramas + ComplementNB
p_bi_nb = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', ComplementNB(alpha=0.5))
])

f1_uni_lr = cross_val_score(p_uni_lr, df['texto'], df['intencion'], cv=cv, scoring='f1_macro').mean()
f1_bi_lr  = cross_val_score(p_bi_lr,  df['texto'], df['intencion'], cv=cv, scoring='f1_macro').mean()
f1_uni_nb = cross_val_score(p_uni_nb, df['texto'], df['intencion'], cv=cv, scoring='f1_macro').mean()
f1_bi_nb  = cross_val_score(p_bi_nb,  df['texto'], df['intencion'], cv=cv, scoring='f1_macro').mean()

print("=== RENDIMIENTO COMPARATIVO (F1-Macro Promedio en 5 Folds) ===")
print(f"1. Regresión Logística - Solo Unigramas (1, 1)        : {f1_uni_lr:.4f}")
print(f"2. Regresión Logística - Unigramas + Bigramas (1, 2)  : {f1_bi_lr:.4f}  <-- Ganador / Producción")
print(f"3. Complement Naive Bayes - Solo Unigramas (1, 1)    : {f1_uni_nb:.4f}")
print(f"4. Complement Naive Bayes - Unigramas + Bigramas (1, 2): {f1_bi_nb:.4f}")

In [ ]:
# Gráfico comparativo de F1-Macro
fig, ax = plt.subplots(figsize=(8, 4))
etiquetas = ['Reg. Logística (1,1)', 'Reg. Logística (1,2)', 'Naive Bayes (1,1)', 'Naive Bayes (1,2)']
puntajes = [f1_uni_lr, f1_bi_lr, f1_uni_nb, f1_bi_nb]
colores = ['#3498db', '#2ecc71', '#9b59b6', '#e67e22']

barras_f1 = ax.bar(etiquetas, puntajes, color=colores, width=0.5, edgecolor='black', linewidth=1)
ax.set_ylim(0, 1.15)
ax.set_ylabel('F1-Score Macro Promedio')
ax.set_title('Impacto de Bigramas en la Clasificación de Intenciones')

for bar in barras_f1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, h + 0.02, f"{h:.3f}", ha='center', va='bottom', fontweight='bold')

ruta_img2 = raiz_proyecto / "artifacts" / "f1_comparativa_modelos.png"
plt.savefig(ruta_img2, dpi=120, bbox_inches="tight")
plt.tight_layout()
print(f"Gráfico guardado en: {ruta_img2}")

## 7. Experimento 4 (Bonus): Comparativa Conceptual TF-IDF vs. Embeddings

Para responder a posibles preguntas teóricas del docente durante la defensa individual:

In [ ]:
tabla_embeddings = pd.DataFrame([
    {"Criterio": "Tipo de representación", "TF-IDF (Producción)": "Dispersa (Sparse), vocabulario de palabras", "Embeddings Semánticos": "Densa (Dense), vectores continuos (300d)"},
    {"Criterio": "Comprensión de sinónimos", "TF-IDF (Producción)": "No directa (ortográfica/léxica)", "Embeddings Semánticos": "Sí (distancia semántica en el espacio)"},
    {"Criterio": "Tiempo de entrenamiento", "TF-IDF (Producción)": "< 1 segundo (óptimo para modificación en vivo)", "Embeddings Semánticos": "Más lento, requiere modelos pesados"},
    {"Criterio": "Tamaño del artefacto", "TF-IDF (Producción)": "< 500 KB (ideal para API ágil)", "Embeddings Semánticos": "45 MB a 500 MB (spaCy / BERT)"},
    {"Criterio": "Interpretabilidad", "TF-IDF (Producción)": "Total (coeficiente directo por término)", "Embeddings Semánticos": "Baja (caja negra de activaciones neuronales)"},
    {"Criterio": "Idoneidad para el proyecto", "TF-IDF (Producción)": "Excelente para consultas cortas con palabras clave", "Embeddings Semánticos": "Riesgo de dilución de palabras determinantes"}
])

tabla_embeddings

## 8. Conclusiones para la Defensa Individual

1. **Elección de TF-IDF:** Es la representación óptima para un corpus de consultas específicas de tiendas de ropa, balanceando interpretabilidad, rapidez y precisión.
2. **Justificación de Bigramas `(1, 2)`:** Mejora el $F_1$-Macro al capturar el contexto directo de expresiones compuestas como `'talla 32'`, `'sin stock'` o `'envío gratis'`.
3. **Justificación de `sublinear_tf=True`:** Atenúa la repetición de palabras en consultas informales de chat.
4. **Modularidad del Pipeline:** La función `crear_vectorizador(config)` en `src/nlp/vectorizacion.py` se integra limpiamente en el `Pipeline` de scikit-learn junto al clasificador en `train.py`.